# 09 – Person Voice Works

Esplorazione e data cleaning del dataset `person_voice_works.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `person_mal_id` | ID univoco della persona (doppiatore/attrice) su MyAnimeList |
| `role` | Tipo di ruolo |
| `anime_mal_id` | ID univoco dell'anime su MAL |
| `character_mal_id` | ID univoco del personaggio doppiato su MAL |
| `language` | Lingua del doppiaggio |

## 1. Import e caricamento dati
Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [27]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_pvw = pd.read_csv('../datasets/person_voice_works.csv')
print(f'Shape: {df_pvw.shape}')
df_pvw.info()
df_pvw.head()

Shape: (489516, 5)
<class 'pandas.DataFrame'>
RangeIndex: 489516 entries, 0 to 489515
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   person_mal_id     489516 non-null  int64
 1   role              489516 non-null  str  
 2   anime_mal_id      489516 non-null  int64
 3   character_mal_id  489516 non-null  int64
 4   language          489516 non-null  str  
dtypes: int64(3), str(2)
memory usage: 18.7 MB


,person_mal_id,role,anime_mal_id,character_mal_id,language
0,1,Main,55830,2514,Japanese
1,1,Supporting,60602,2822,Japanese
2,1,Supporting,59229,140499,Japanese
3,1,Supporting,60427,275856,Japanese
4,1,Supporting,62067,190335,Japanese


**Osservazioni iniziali:**
- Il dataset contiene 489,516 righe e tutte le colonne sono popolate e non contengono valori nulli.
- I tipi di dati sono adeguati.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [28]:
n_originale = len(df_pvw)

mask_dup = df_pvw.duplicated(keep=False)
n_righe_coinvolte = mask_dup.sum()
n_gruppi = df_pvw[mask_dup].duplicated(keep='first').sum()
n_tenute = n_righe_coinvolte - n_gruppi

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_pvw.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_pvw):,}')

Righe totali coinvolte in duplicazioni : 1,488
  → prime occorrenze mantenute         : 537
  → occorrenze extra rimosse           : 951

Righe prima della rimozione : 489,516
Righe dopo la rimozione     : 488,565


**Osservazioni:**

**1,488 righe** risultano coinvolte in duplicazioni esatte su tutte le colonne. Di queste, **537** sono prime occorrenze mantenute e **951** occorrenze extra rimosse. Dopo la rimozione il dataset scende da 489,516 a **488,565 righe**.

Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `person_mal_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria di `person_details.csv`.

I valori duplicati sono **attesi**: lo stesso doppiatore può aver lavorato a più anime/personaggi in lingue diverse.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `person_details_clean.csv`.

Usiamo quindi `check_fk` al posto di `analyze`, che effettua entrambi i controlli.

In [29]:
df_persons = pd.read_csv('../datasets_cleaned/person_details_clean.csv')

mask_orphan_person = check_fk(df_pvw['person_mal_id'], df_persons['person_mal_id'], child_df=df_pvw)

print(f'Null in person_mal_id               : {df_pvw["person_mal_id"].isna().sum()}')
print(f'Duplicati in person_mal_id (attesi) : {df_pvw["person_mal_id"].duplicated().sum():,}')


  Colonna FK  (tabella figlia)        person_mal_id
  Colonna PK  (tabella padre)         person_mal_id
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       488,565
  Valori null nella FK                0  (0.00%)
  Valori non-null nella FK            488,565  (100.00%)
  Valori unici nella PK padre         75,211

  ✓  Righe con FK valida              488,565  (100.00%)
  ✗  Righe orfane (FK non in PK)      0  (0.00%)
     → ID orfani unici                0

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

Nessuna violazione. Tutti i valori FK esistono nella tabella padre.

Null in person_mal_id               : 0
Duplicati in person_mal_id (attesi) : 466,290


**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID di riferimento valido.
- **Integrità referenziale**: non ci sono righe orfane.

Nessuna pulizia necessaria.

### 2.2 `role`

Colonna che indica il tipo di ruolo del doppiatore. Ci interessa verificare la presenza di valori nulli e anomalie. Per questo utilizziamo `analyze`. I duplicati sono **attesi**.

In [31]:
df_pvw['role'] = df_pvw['role'].str.strip()
analyze(df_pvw['role'])


  Nome serie                     role
  dtype                          str
  Dimensione                     488,565
  Range indice                   0 … 489515
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   488,565
  Valori non nulli               488,565  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   2  (0.00%)
  Valori duplicati               488,563  (100.00%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  4  caratteri
  Lunghezza max                  10  caratteri
  Lunghezza media                8.78  caratteri
  Lunghezza mediana              10.0000  caratteri
  Dev. standard lunghezza        2.41
  IQR lunghezza                  0.0000

Valori di lunghe

**Osservazioni:**
- Nessun null. Tutte le righe hanno un valore per il ruolo.
- Solo due valori distinti (`Main` e `Supporting`).

**Nessuna pulizia necessaria.**

### 2.3 `anime_mal_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria `mal_id` di `details.csv`.

I valori duplicati sono **attesi**: ogni anime coinvolge più doppiatori per più personaggi.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `details_clean.csv`.

Usiamo quindi `check_fk` al posto di `analyze`, che effettua entrambi i controlli.

In [32]:
df_details = pd.read_csv('../datasets_cleaned/details_clean.csv')

mask_orphan_anime = check_fk(df_pvw['anime_mal_id'], df_details['mal_id'], child_df=df_pvw)

print(f'Null in anime_mal_id               : {df_pvw["anime_mal_id"].isna().sum()}')
print(f'Duplicati in anime_mal_id (attesi) : {df_pvw["anime_mal_id"].duplicated().sum():,}')


  Colonna FK  (tabella figlia)        anime_mal_id
  Colonna PK  (tabella padre)         mal_id
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       488,565
  Valori null nella FK                0  (0.00%)
  Valori non-null nella FK            488,565  (100.00%)
  Valori unici nella PK padre         28,955

  ✓  Righe con FK valida              488,565  (100.00%)
  ✗  Righe orfane (FK non in PK)      0  (0.00%)
     → ID orfani unici                0

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

Nessuna violazione. Tutti i valori FK esistono nella tabella padre.

Null in anime_mal_id               : 0
Duplicati in anime_mal_id (attesi) : 476,123


**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID anime valido.
- **Integrità referenziale**: non ci sono righe orfane.

Nessuna pulizia necessaria. 

### 2.4 `character_mal_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria `character_mal_id` di `characters.csv`.

I valori duplicati sono **attesi**: lo stesso personaggio può essere doppiato da più persone in lingue diverse.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `characters_clean.csv`.

Usiamo quindi `check_fk` al posto di `analyze`, che effettua entrambi i controlli.

In [34]:
df_characters = pd.read_csv('../datasets_cleaned/characters_clean.csv')

mask_orphan_char = check_fk(df_pvw['character_mal_id'], df_characters['character_mal_id'], child_df=df_pvw)

print(f'Null in character_mal_id               : {df_pvw["character_mal_id"].isna().sum()}')
print(f'Duplicati in character_mal_id (attesi) : {df_pvw["character_mal_id"].duplicated().sum():,}')


  Colonna FK  (tabella figlia)        character_mal_id
  Colonna PK  (tabella padre)         character_mal_id
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       488,565
  Valori null nella FK                0  (0.00%)
  Valori non-null nella FK            488,565  (100.00%)
  Valori unici nella PK padre         209,506

  ✓  Righe con FK valida              488,565  (100.00%)
  ✗  Righe orfane (FK non in PK)      0  (0.00%)
     → ID orfani unici                0

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

Nessuna violazione. Tutti i valori FK esistono nella tabella padre.

Null in character_mal_id               : 0
Duplicati in character_mal_id (attesi) : 355,574


**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID personaggio valido.
- **Integrità referenziale**: non ci sono righe orfane.

Nessuna pulizia necessaria. 

### 2.5 `language`

Colonna che indica la lingua del doppiaggio. Ci interessa verificare la presenza di valori nulli e anomalie. Per questo utilizziamo `analyze`. I duplicati sono **attesi**.

In [36]:
df_pvw['language'] = df_pvw['language'].str.strip()
analyze(df_pvw['language'])


  Nome serie                     language
  dtype                          str
  Dimensione                     488,565
  Range indice                   0 … 489515
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   488,565
  Valori non nulli               488,565  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   11  (0.00%)
  Valori duplicati               488,554  (100.00%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  6  caratteri
  Lunghezza max                  15  caratteri
  Lunghezza media                7.87  caratteri
  Lunghezza mediana              8.0000  caratteri
  Dev. standard lunghezza        2.17
  IQR lunghezza                  1.0000

Valori di lu

**Osservazioni:**
- Nessun null. Tutte le righe hanno un valore per la lingua.
- Ci sono 11 valori unici che corrispondono alle lingue nelle quali gli anime sono disponibili. 
- Il giapponese è la lingua dominante. È atteso in quanto MAL è orientato agli anime giapponesi, seguito dall'inglese.

**Nessuna pulizia necessaria.**

### 2.6 Chiave primaria `(person_mal_id, anime_mal_id, character_mal_id, language)`

La chiave primaria è la quadrupla `(person_mal_id, anime_mal_id, character_mal_id, language)`: uno stesso doppiatore può interpretare lo stesso personaggio nello stesso anime in lingue diverse, ma non due volte nella stessa lingua. Il campo `role` è un attributo descrittivo, non parte della chiave.

Verifichiamo che questa combinazione sia univoca dopo la pulizia.

In [37]:
pk_cols = ['person_mal_id', 'anime_mal_id', 'character_mal_id', 'language']
n_pk_dup = df_pvw.duplicated(subset=pk_cols, keep=False).sum()
print(f'Duplicati sulla chiave primaria {pk_cols}: {n_pk_dup}')

if n_pk_dup > 0:
    print('→ Rimozione duplicati sulla chiave primaria...')
    df_pvw.drop_duplicates(subset=pk_cols, keep='first', inplace=True)
    print(f'Righe dopo la rimozione: {len(df_pvw):,}')
else:
    print('→ Chiave primaria già univoca, nessuna operazione richiesta.')

Duplicati sulla chiave primaria ['person_mal_id', 'anime_mal_id', 'character_mal_id', 'language']: 0
→ Chiave primaria già univoca, nessuna operazione richiesta.


La chiave primaria è univoca dopo la pulizia. Il dataset è pronto per il salvataggio.

## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effettuiamo il salvataggio del dataset finale.

In [38]:
print('Riepilogo Dataset Pulito')
print(f'Righe originali      : {n_originale:>10,}')
print(f'Righe dopo cleaning  : {len(df_pvw):>10,}')
print(f'Righe rimosse totali : {n_originale - len(df_pvw):>10,}')
print()
df_pvw.to_csv('../datasets_cleaned/person_voice_works_clean.csv', index=False)
print('Salvato: datasets_cleaned/person_voice_works_clean.csv')

Riepilogo Dataset Pulito
Righe originali      :    489,516
Righe dopo cleaning  :    488,565
Righe rimosse totali :        951

Salvato: datasets_cleaned/person_voice_works_clean.csv
